# Day 2 – Put the models to the test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kouroshSA/nanoGPT-Shakespeare-Simpsons-Lab/blob/main/colab/Day2_Inference_Lab.ipynb)

Plan for today (about 60–75 minutes):

| | | time |
|---|---|---|
| **Part A** (Model S, Shakespeare only) | **A1** same prompt through three snapshots (after 1,000, 5,000 and 10,000 training steps); **A2** one snapshot at five "temperatures" | ~25 min |
| **Part B** (Model M, Shakespeare + Simpsons) | **B1** and **B2** the same two steps: can the model keep the two styles apart? | ~25 min |
| **Wrap-up** | answer the questions in your report | ~10–15 min |

**How you evaluate the writing:** Shakespeare is not easy to read, so you will ask an AI (**Gemini**, at gemini.google.com) to judge the model's writing.
The notebook builds a ready-made prompt for you: copy it, paste it into Gemini, and read what it says. Always check the AI's answer against the text yourself — AI can be wrong too.

*Made by Claude AI (Claude Sonnet 5.5), guided by K. Salehi-Ashtiani.*

## 0. Setup

In [ ]:
# ── Setup (run this first) ───────────────────────────────────────────────────
import os, sys, subprocess

REPO_URL = "https://github.com/kouroshSA/nanoGPT-Shakespeare-Simpsons-Lab"
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 1. get the code (in Colab) -------------------------------------------------
if IN_COLAB and not os.path.exists("train.py"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/lab"], check=True)
    os.chdir("/content/lab")
print("working in:", os.getcwd())

# 2. where do the results live?  Colab erases everything when the session ends, so we use Google Drive ---
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    LAB_DIR = "/content/drive/MyDrive/nanoGPT_lab"
else:
    LAB_DIR = os.path.abspath("lab_runs")            # running on your own computer
os.makedirs(LAB_DIR, exist_ok=True)
print("results folder:", LAB_DIR)

def run(cmd):
    """Run a command and show its output live."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (the CPU works too, just slower)")
from lab_tools import *

# the checkpoints saved on Day 1
FOLDERS = {"shakespeare": f"{LAB_DIR}/shakespeare", "mix": f"{LAB_DIR}/mix"}
for k, f in FOLDERS.items():
    try:
        print(f"{k:12s}", len(checkpoints(f)), "checkpoints found")
    except FileNotFoundError as e:
        print(f"{k:12s} ⚠️", e)

## Part A – Model S (Shakespeare only)

### A1. The same prompt, three snapshots (1,000 / 5,000 / 10,000 steps)
A **prompt** is the start of the text. The model continues it for 1,000 characters.
The random numbers are the same for every snapshot (`which=(1, 5, 10)` picks snapshots 1, 5 and 10; use `which=None` to see all ten), so any difference comes from *how long the model trained*.
**Temperature** 0.8 is a balanced setting (low = careful and repetitive, high = adventurous and messy).

Change the prompt if you like — try `ROMEO.\n`, `Shall I compare thee to`, or a line of your own.

In [ ]:
PROMPT_A = "HAMLET.\n"
resA1 = compare_checkpoints(FOLDERS["shakespeare"], PROMPT_A, n_chars=1000, temperature=0.8)
save_results(resA1, f"{LAB_DIR}/A1_results.txt");

**Look first, then ask Gemini.** Write down in your report what changes between the first and last snapshot (spelling? words? layout? meaning?).
Then run the next cell, copy the **whole** text it prints, and paste it into Gemini.

In [ ]:
FIRST, LAST = "ckpt_001", "ckpt_010"      # you can also try "ckpt_005" against "ckpt_010"
gp = gemini_prompt_from(resA1, FIRST, LAST, "shakespeare")

### A2. One snapshot, five temperatures
Now we use only the **last** snapshot and change the temperature from 0.1 to 1.5. Same prompt, same random numbers.

In [ ]:
resA2 = temperature_sweep(FOLDERS["shakespeare"], PROMPT_A, n_chars=1000)   # temperatures 0.1, 0.5, 0.8, 1.1, 1.5
save_results(resA2, f"{LAB_DIR}/A2_results.txt");

In [ ]:
gp = gemini_prompt_from(resA2, "T=0.1", "T=1.5", "shakespeare")   # compare the coolest and the hottest; try other pairs too

**Questions for your report (Part A):**
1. What did the model learn *first* (spelling, line layout, character names, real words, grammar, meaning)? Which snapshot shows each?
2. What did Gemini say about the first vs. the last snapshot? Do you agree? Give one example where Gemini was right and, if you can find one, one where it was wrong or too generous.
3. How does temperature change the writing? Which temperature would you choose for a poem? For a quiz answer? Why?
4. Does the model *understand* Shakespeare? What evidence in the text supports your answer?

## Part B – Model M (Shakespeare + Simpsons)

Model M read both Shakespeare and Simpsons dialogue, mixed together, and was **never told** which is which.
**Question: can it keep the two styles separate, or does it blend them?**

### B1. A neutral prompt, three snapshots
This prompt belongs to neither style, so the model has to *choose*. Then try a Shakespeare prompt and a Simpsons prompt too.

In [ ]:
PROMPT_B = "Yesterday I went to the store and"
resB1 = compare_checkpoints(FOLDERS["mix"], PROMPT_B, n_chars=1000, temperature=0.8)
save_results(resB1, f"{LAB_DIR}/B1_results.txt");

In [ ]:
gp = gemini_prompt_from(resB1, "ckpt_001", "ckpt_010", "mix")

Try your own prompts that lean towards each style (run each as often as you like — change the `seed` to get a different text):

In [ ]:
last = checkpoints(FOLDERS["mix"])[-1]
for prompt in ["HAMLET.\n", "Marge, where are my keys?\n", "Homer: To be, or not to be,"]:
    print("=" * 70, "\nPROMPT:", repr(prompt), "\n" + "=" * 70)
    print(generate(last, prompt, n_chars=500, temperature=0.8, seed=1))

### B2. The last snapshot at five temperatures

In [ ]:
resB2 = temperature_sweep(FOLDERS["mix"], PROMPT_B, n_chars=1000)
save_results(resB2, f"{LAB_DIR}/B2_results.txt");

In [ ]:
gp = gemini_prompt_from(resB2, "T=0.1", "T=1.5", "mix")

**Questions for your report (Part B):**
5. Does Model M stay in **one** style within a passage, or does it jump between Shakespeare and Simpsons? Does it commit to one style already at 5,000 steps, or only at 10,000? Quote the evidence (and Gemini's view).
6. Does the temperature change *which* style it picks, or only how messy it is?
7. Compare Part A and Part B: Model S saw only Shakespeare. Does Model S write *better* Shakespeare than Model M? Why might that be?
8. Reflect: what is this tiny model doing that is similar to a big chatbot, and what is it not doing?

## Optional – a chat window
Runs a small chat-style window where you can talk to any snapshot of either model. (It continues a script rather than answering questions.)

In [ ]:
if IN_COLAB:
    run("pip -q install 'gradio>=5,<6'")
which = "shakespeare"   # or "mix"
print("Starting… open the public link that appears (it ends in .gradio.live) and keep this cell running.")
run(f"python chat_ui.py --out_dir={FOLDERS[which]} --share=True")